# Notebook 3: Nhận dạng tiếng nói (PhoWhisper-small) có lưu tiến trình

Chạy PhoWhisper trên một tập con **ngẫu nhiên, cố định** của audio, lưu kết quả **theo từng nhóm** (`chunk_00000.csv`, ...). Phiên Colab nào bị ngắt thì phiên sau chạy lại notebook này và **chỉ làm tiếp các nhóm còn thiếu**.

**Cách chọn mẫu:** hoán vị ngẫu nhiên (SEED cố định) của các dòng có audio, lưu trong `order.csv`. Vị trí `[0, N_DEV)` là tập **dev** (5,500), vị trí `[N_DEV, N_DEV+N_TRAIN)` là tập **train** (49,500); tổng 55,000, chia 90/10. Vì chạy theo thứ tự hoán vị, mọi tiền tố của tập train đều là một tập con ngẫu nhiên hợp lệ, nên các tập con lồng nhau 0.25 / 0.5 / 1.0 là 25% / 50% / 100% đầu của train.

**Đường dẫn:** mọi đường dẫn đều tương đối so với `ROOT`. Đặt `ROOT` là thư mục làm việc của bạn. `OUT_DIR` phải nằm trên Drive thì tiến trình mới còn sau khi phiên kết thúc; `AUDIO_DIR` nên là bản giải nén trên ổ Colab (đọc nhanh).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install -U transformers accelerate soundfile librosa

In [ ]:
import os, json, time, random, shutil
import numpy as np, pandas as pd, soundfile as sf, librosa, torch
from concurrent.futures import ThreadPoolExecutor

# ---------- ĐƯỜNG DẪN (tương đối so với ROOT; thay ROOT bằng thư mục làm việc của bạn) ----------
ROOT      = '/content/drive/MyDrive/project_DPL'
META_CSV  = os.path.join(ROOT, 'datasets/vietspeech_100k/metadata_with_en_vinai.csv')   # cột: id, audio_path, vi_text, en_text
AUDIO_DIR = os.path.join(ROOT, 'datasets/vietspeech_100k/audio')   # thư mục audio đã giải nén (trên Drive)
MODEL_DIR = os.path.join(ROOT, 'models/PhoWhisper-small')   # chưa có thì tải một lần từ 'vinai/PhoWhisper-small' rồi lưu vào đây
OUT_DIR   = os.path.join(ROOT, 'results/asr_out')                    # kết quả; PHẢI nằm trên Drive để giữ qua các phiên
ASR_MODEL = 'vinai/PhoWhisper-small'

# ---------- THAM SỐ ----------
SEED        = 42
N_DEV       = 5500       # tập val = 10% của 55,000 mẫu (chốt: train 49,500 + val 5,500); val cũng chạy ASR để giống lúc test
N_TRAIN     = 49500      # số mẫu train (chốt). Các nhóm đã xong vẫn dùng lại nếu sau này tăng số này
CHUNK       = 1000       # số mẫu mỗi nhóm lưu một file
ASR_BATCH   = 8          # batch đã đo ở notebook đo tài nguyên
READ_WORKERS = 8
MAX_SEC     = 30         # Whisper nhận tối đa 30 s; audio dài hơn bị cắt (có ghi cột dur_s)
TIME_LIMIT_H = None      # None = chạy đến hết (nếu phiên Colab bị ngắt, chạy lại notebook để làm tiếp). Hoặc đặt số giờ để dừng êm sau nhóm hiện tại
SR          = 16000
NUM_BEAM = 3                 # Beam của ASR

os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU (rất chậm)')

In [ ]:
# 1) Thứ tự mẫu: tạo MỘT LẦN, lưu lại, các phiên sau dùng lại (đảm bảo tập con ổn định)
def fname(p): return os.path.basename(str(p).replace('\\', '/'))

df = pd.read_csv(META_CSV)
for c in ['id', 'audio_path', 'vi_text', 'en_text']: assert c in df.columns, f'Thiếu cột {c}'
assert df['id'].is_unique, 'Cột id không duy nhất'
df = df[df.vi_text.notna() & (df.vi_text.astype(str).str.strip() != '') &
        df.en_text.notna() & (df.en_text.astype(str).str.strip() != '')].reset_index(drop=True)
df['audio_file'] = df.audio_path.map(fname)
df = df.set_index('id', drop=False)

ORDER_CSV, META_JSON = os.path.join(OUT_DIR, 'order.csv'), os.path.join(OUT_DIR, 'order_meta.json')
if os.path.exists(ORDER_CSV) and os.path.exists(META_JSON):
    om = json.load(open(META_JSON))
    assert om['seed'] == SEED and om['n_dev'] == N_DEV, (
        f'order.csv được tạo với seed={om["seed"]}, n_dev={om["n_dev"]}; khác cấu hình hiện tại. '
        'Đừng đổi SEED/N_DEV giữa chừng (sẽ làm lẫn dev và train). Muốn làm lại từ đầu thì xóa thư mục OUT_DIR.')
    order = pd.read_csv(ORDER_CSV)
    print(f'Dùng lại order.csv ({len(order):,} dòng).')
else:
    # TỐI ƯU: Không quét toàn bộ thư mục bằng os.path.exists đa luồng trên Drive nữa để tránh lỗi I/O
    print('Đang kiểm tra nhanh đường dẫn audio...')
    first_audio = os.path.join(AUDIO_DIR, df['audio_file'].iloc[0])
    assert os.path.exists(first_audio), f'Không tìm thấy file audio đầu tiên tại {first_audio}. Vui lòng kiểm tra lại AUDIO_DIR.'

    # Coi như tất cả file trong metadata đều có sẵn để tránh quét Drive
    avail = df
    print(f'Nhận dạng cấu hình với {len(avail):,} mẫu.')

    perm = np.random.RandomState(SEED).permutation(len(avail))
    order = pd.DataFrame({'pos': np.arange(len(avail)), 'id': avail['id'].values[perm]})
    order['split'] = np.where(order.pos < N_DEV, 'dev', 'train')
    tmp = ORDER_CSV + '.tmp'; order.to_csv(tmp, index=False); os.replace(tmp, ORDER_CSV)
    json.dump({'seed': SEED, 'n_dev': N_DEV, 'n_avail': int(len(avail))}, open(META_JSON, 'w'))
    print('Đã tạo order.csv')

TOTAL = len(order) if N_TRAIN is None else N_DEV + N_TRAIN
if TOTAL > len(order):
    print(f'CẢNH BÁO: cần {TOTAL:,} mẫu nhưng chỉ có {len(order):,} có audio; giảm còn {len(order):,}.')
    TOTAL = len(order)
order = order.iloc[:TOTAL]

# TỐI ƯU: Bỏ hoàn toàn việc dùng os.scandir và ThreadPoolExecutor.exists để tránh nghẽn I/O trên Google Drive
print('KIỂM TRA DỮ LIỆU: Bỏ qua kiểm tra quét file trên Drive để tránh lỗi OSError [Errno 5].')
N_CHUNKS = (TOTAL + CHUNK - 1) // CHUNK
print(f'Sẽ nhận dạng {TOTAL:,} mẫu (dev {min(N_DEV, TOTAL):,} + train {max(0, TOTAL-N_DEV):,}) trong {N_CHUNKS} nhóm.')

In [ ]:
# Audio theo từng zip (chia theo vị trí trong order.csv: nhóm k nằm trong zip k // PART_CHUNKS)
# - Trên runtime CPU: tạo đủ các zip còn thiếu rồi dừng (chỉ cần làm một lần).
# - Trên runtime GPU: ô chạy nhóm gọi ensure_part(p) để copy + giải nén đúng zip cần, và chuẩn bị sẵn zip kế tiếp trong nền.
import zipfile, threading
from math import ceil

PART_CHUNKS   = 5                                    # 5 nhóm = 5,000 file mỗi zip
PART_SIZE     = PART_CHUNKS * CHUNK
PACK_WORKERS  = READ_WORKERS                         # số luồng đọc Drive khi đóng gói
AUDIO_DRIVE   = os.path.join(ROOT, 'datasets/vietspeech_100k/audio')        # audio gốc đã giải nén trên Drive
ZIP_DIR_DRIVE = os.path.join(ROOT, 'datasets/vietspeech_100k/audio_zips')   # nơi lưu các zip (giữ qua các phiên)
ZIP_DIR_LOCAL = '/content/audio_zips'
AUDIO_LOCAL   = '/content/audio'
ZIP_PREFIX    = f'audio_s{SEED}_dev{N_DEV}_tr{N_TRAIN}'                      # mã định danh gắn với seed và cấu hình chia
BUILD_ONLY    = (DEVICE == 'cpu')                    # runtime CPU: chỉ tạo zip

N_PARTS = ceil(TOTAL / PART_SIZE)
for d in (ZIP_DIR_DRIVE, ZIP_DIR_LOCAL, AUDIO_LOCAL): os.makedirs(d, exist_ok=True)

def part_files(p):
    return df.loc[order.id.iloc[p * PART_SIZE:(p + 1) * PART_SIZE], 'audio_file'].tolist()
def zip_name(p): return f'{ZIP_PREFIX}_p{p:02d}.zip'
def zip_ok(path, files):
    try:
        with zipfile.ZipFile(path) as z: names = z.namelist()
        return len(names) == len(files) and set(names) == set(files)
    except Exception:
        return False
def _free(path): return shutil.disk_usage(path).free / 2**30

def _copy_atomic(src, dst):
    tmp = dst + '.part'
    shutil.copyfile(src, tmp)
    assert os.path.getsize(tmp) == os.path.getsize(src), f'Chép sai kích thước: {src}'
    os.replace(tmp, dst)

def make_part(p):
    # Tạo zip số p từ thư mục audio trên Drive (chậm). Ghi local trước, rồi chép lên Drive.
    files = part_files(p); name = zip_name(p)
    zl, zd = os.path.join(ZIP_DIR_LOCAL, name), os.path.join(ZIP_DIR_DRIVE, name)
    t0 = time.time()
    def rd(f):
        err = None
        for a in range(3):
            try:
                with open(os.path.join(AUDIO_DRIVE, f), 'rb') as fh: return f, fh.read()
            except Exception as e:
                err = e; time.sleep(1 + a)
        return f, None
    missing = []
    with zipfile.ZipFile(zl + '.part', 'w', zipfile.ZIP_STORED) as z, ThreadPoolExecutor(PACK_WORKERS) as ex:
        for i in range(0, len(files), 200):                       # theo lô nhỏ để không giữ cả zip trong RAM
            for f, b in ex.map(rd, files[i:i + 200]):
                if b is None: missing.append(f)
                else: z.writestr(zipfile.ZipInfo(f), b, compress_type=zipfile.ZIP_STORED)
            if (i // 200) % 5 == 4: print(f'   zip {p}: {i + 200:,}/{len(files):,} file ({time.time() - t0:.0f}s)')
    if missing:
        os.remove(zl + '.part')
        raise FileNotFoundError(f'Zip {p}: thiếu {len(missing)} file audio trên Drive, ví dụ {missing[:3]}')
    os.replace(zl + '.part', zl)
    assert zip_ok(zl, files), f'Zip {p} vừa tạo không hợp lệ'
    _copy_atomic(zl, zd)
    print(f'Đã tạo {name}: {os.path.getsize(zl) / 2**20:.0f} MB, {len(files):,} file, {time.time() - t0:.0f}s | ổ local trống {_free("/content"):.1f} GB')
    if BUILD_ONLY: os.remove(zl)                                  # CPU: không cần giữ bản local

def _prepare(p):
    files = part_files(p)
    if all(os.path.exists(os.path.join(AUDIO_LOCAL, f)) for f in files): return
    name = zip_name(p); zl, zd = os.path.join(ZIP_DIR_LOCAL, name), os.path.join(ZIP_DIR_DRIVE, name)
    t0 = time.time()
    if not (os.path.exists(zl) and zip_ok(zl, files)):
        if os.path.exists(zd) and zip_ok(zd, files):
            _copy_atomic(zd, zl)
        else:
            print(f'CẢNH BÁO: chưa có {name} trên Drive; tạo ngay tại chỗ (chậm). Nên tạo trước bằng runtime CPU.')
            make_part(p)
    n = 0
    with zipfile.ZipFile(zl) as z:
        for f in files:
            dst = os.path.join(AUDIO_LOCAL, f)
            if not os.path.exists(dst):
                with z.open(f) as s, open(dst + '.part', 'wb') as d: shutil.copyfileobj(s, d)   # zipfile kiểm CRC khi đọc
                os.replace(dst + '.part', dst); n += 1
    print(f'Zip {p}: sẵn sàng ({n:,} file giải nén, {time.time() - t0:.0f}s) | ổ local trống {_free("/content"):.1f} GB')

_pool_prefetch = ThreadPoolExecutor(1); _futures = {}
def ensure_part(p):
    # Gọi đầu mỗi nhóm: đảm bảo zip p đã được giải nén local, và chuẩn bị sẵn zip p+1 nếu còn nhóm cần dùng.
    fut = _futures.pop(p, None)
    if fut is not None: fut.result()
    else: _prepare(p)
    q = p + 1
    if q < N_PARTS and q not in _futures:
        ks = range(q * PART_CHUNKS, min((q + 1) * PART_CHUNKS, N_CHUNKS))
        if any(k in TODO_SET for k in ks): _futures[q] = _pool_prefetch.submit(_prepare, q)

if BUILD_ONLY:
    print(f'Runtime CPU: tạo các zip còn thiếu ({N_PARTS} zip, mỗi zip {PART_SIZE:,} file).')
    for p in range(N_PARTS):
        if zip_ok(os.path.join(ZIP_DIR_DRIVE, zip_name(p)), part_files(p)): print(f'Zip {p}: đã có, bỏ qua.')
        else: make_part(p)
    print('XONG: đã có đủ các zip trên Drive. Từ phiên GPU, chạy bình thường: ô chạy nhóm sẽ tự lấy zip cần dùng.')
else:
    print(f'Runtime GPU: audio sẽ được lấy theo từng zip ({N_PARTS} zip). Đọc từ {AUDIO_LOCAL}.')
AUDIO_DIR = AUDIO_LOCAL

In [ ]:
# 2) Nạp mô hình
import os, shutil, torch
from transformers import pipeline

if not os.path.isfile(os.path.join(MODEL_DIR, 'config.json')):
    # tải MỘT LẦN về MODEL_DIR (ghi vào thư mục tạm rồi đổi tên, tránh thư mục dang dở)
    from huggingface_hub import snapshot_download
    tmpd = MODEL_DIR + '.tmp'
    shutil.rmtree(tmpd, ignore_errors=True); os.makedirs(os.path.dirname(MODEL_DIR) or '.', exist_ok=True)
    snapshot_download(ASR_MODEL, local_dir=tmpd, ignore_patterns=['*.msgpack', '*.h5', '*.ot'])
    assert os.path.isfile(os.path.join(tmpd, 'config.json')), 'Tải mô hình không đủ file'
    shutil.rmtree(MODEL_DIR, ignore_errors=True); os.rename(tmpd, MODEL_DIR)
    print('Đã lưu mô hình vào', MODEL_DIR)

src = MODEL_DIR
print('Nạp mô hình từ:', src)

def _make():
    kw = dict(model=src, device=0 if DEVICE == 'cuda' else -1)
    dt = torch.float16 if DEVICE == 'cuda' else torch.float32
    try: return pipeline('automatic-speech-recognition', dtype=dt, **kw)
    except TypeError: return pipeline('automatic-speech-recognition', torch_dtype=dt, **kw)

PIPE = _make()
# Bổ sung num_beams=2 hoặc beam=3 để tối ưu hóa tốc độ giải mã thay vì mặc định là beam=5
GK = {'language': 'vi', 'task': 'transcribe', 'num_beams': NUM_BEAM}

def transcribe_dataset(wavs_list):
    # Sử dụng generator để truyền dữ liệu tối ưu vào pipeline mà không bị lỗi tương thích Dataset
    def gen():
        for w in wavs_list:
            yield {'raw': w, 'sampling_rate': SR}
    out = PIPE(gen(), batch_size=ASR_BATCH, generate_kwargs=GK)
    return [o['text'].strip() for o in out]

# Khởi động mô hình
_ = transcribe_dataset([np.zeros(SR, dtype='float32')] * 2)
print('Sẵn sàng.')

Nạp mô hình từ: /content/drive/MyDrive/project_DPL/models/PhoWhisper-small


Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

Sẵn sàng.


In [ ]:
import gc
from torch.utils.data import Dataset

# Định nghĩa một Dataset chuẩn cho Hugging Face Pipeline
class AudioDataset(Dataset):
    def __init__(self, wavs, srs):
        self.wavs = wavs
        self.srs = srs
    def __len__(self):
        return len(self.wavs)
    def __getitem__(self, idx):
        return {'raw': self.wavs[idx], 'sampling_rate': self.srs[idx]}

# 3) Chạy theo từng nhóm; nhóm nào đã có và hợp lệ thì bỏ qua
def read_wav(fn):
    try:
        w, sr = sf.read(os.path.join(AUDIO_DIR, fn), dtype='float32')
        if w.ndim > 1:
            w = w.mean(axis=1)
        if len(w) == 0:
            return None, 0.0, 'empty', SR
        max_len = sr * MAX_SEC
        return w[:max_len], len(w) / sr, '', sr
    except Exception as e:
        return None, 0.0, f'{type(e).__name__}: {str(e)[:60]}', SR

def chunk_path(k): return os.path.join(OUT_DIR, f'chunk_{k:05d}.csv')
def chunk_ids(k): return order.id.iloc[k*CHUNK:(k+1)*CHUNK].tolist()

def chunk_ok(k):
    p = chunk_path(k)
    if not os.path.exists(p): return False
    try: return pd.read_csv(p, usecols=['id']).id.tolist() == chunk_ids(k)
    except Exception: return False

done = [k for k in range(N_CHUNKS) if chunk_ok(k)]
todo = [k for k in range(N_CHUNKS) if k not in set(done)]
print(f'Đã xong {len(done)}/{N_CHUNKS} nhóm; còn {len(todo)} nhóm.')
TODO_SET = set(todo)

OPTI_BATCH = 24
def transcribe_dataset_fast(wavs_list, srs_list):
    # Sử dụng AudioDataset chuẩn để giải quyết triệt để cảnh báo của Hugging Face
    dataset = AudioDataset(wavs_list, srs_list)
    with torch.no_grad():
        out = PIPE(dataset, batch_size=OPTI_BATCH, generate_kwargs=GK)
        res = [o['text'].strip() for o in out]
    return res

t_start = time.time(); n_done_now = 0
pool = ThreadPoolExecutor(READ_WORKERS)
MICRO_BATCH_SIZE = 100 # Xử lý nội bộ 100 mẫu một lần để giải phóng RAM

for k in todo:
    if TIME_LIMIT_H is not None and (time.time() - t_start) > TIME_LIMIT_H * 3600:
        print(f'Hết TIME_LIMIT_H={TIME_LIMIT_H}; dừng. Chạy lại notebook ở phiên sau để làm tiếp.'); break
    t0 = time.time()
    rows = df.loc[chunk_ids(k)]
    ensure_part(k // PART_CHUNKS)   # lấy zip audio của nhóm này (và chuẩn bị zip kế tiếp)

    hyp = [''] * len(rows)
    durs = [0.0] * len(rows)
    errors = [''] * len(rows)

    t_read_total = 0.0
    t_asr_total = 0.0

    # Chia nhỏ chunk 1000 mẫu thành các micro-batch để tiết kiệm RAM
    for start_idx in range(0, len(rows), MICRO_BATCH_SIZE):
        sub_rows = rows.iloc[start_idx : start_idx + MICRO_BATCH_SIZE]

        # 1. Đọc song song micro-batch
        t1 = time.time()
        reads = list(pool.map(read_wav, sub_rows.audio_file))
        t_read_total += time.time() - t1

        # Phân loại
        ok_idx = [i for i, r in enumerate(reads) if r[0] is not None]
        for idx, r in enumerate(reads):
            durs[start_idx + idx] = round(r[1], 3)
            errors[start_idx + idx] = r[2]

        # 2. Nhận dạng micro-batch
        if ok_idx:
            wavs = [reads[i][0] for i in ok_idx]
            srs = [reads[i][3] for i in ok_idx]
            t2 = time.time()
            predictions = transcribe_dataset_fast(wavs, srs)
            t_asr_total += time.time() - t2

            for i, text in zip(ok_idx, predictions):
                hyp[start_idx + i] = text

        # Giải phóng vùng nhớ của micro-batch ngay lập tức
        del reads, ok_idx
        gc.collect()

    # Ghi file kết quả cho cả chunk (vẫn đầy đủ 1000 mẫu)
    res = pd.DataFrame({
        'id': rows.id.values,
        'audio_file': rows.audio_file.values,
        'vi_text': rows.vi_text.values,
        'en_text': rows.en_text.values,
        'asr_text': hyp,
        'dur_s': durs,
        'error': errors
    })

    tmp = chunk_path(k) + '.tmp'; res.to_csv(tmp, index=False); os.replace(tmp, chunk_path(k))
    assert chunk_ok(k), f'Nhóm {k} ghi xong nhưng đọc lại không khớp'

    n_done_now += 1
    el = time.time() - t_start; left = len(todo) - n_done_now
    print(f'nhóm {k:3d} | {len(rows)} mẫu | lỗi đọc {int((res.error != "").sum())} | {time.time()-t0:.1f}s '
          f'(đọc {t_read_total:.0f}s | ASR {t_asr_total:.0f}s) | còn {left} nhóm ~{left * el / n_done_now / 3600:.2f} h')

    # Dọn dẹp RAM cuối chunk
    del rows, res, hyp, durs, errors
    gc.collect()

print('Xong phiên này.' if len([k for k in range(N_CHUNKS) if chunk_ok(k)]) < N_CHUNKS else 'ĐÃ XONG TẤT CẢ các nhóm.')

Đã xong 28/55 nhóm; còn 27 nhóm.
nhóm  28 | 1000 mẫu | lỗi đọc 0 | 283.2s (đọc 1s | ASR 280s) | còn 26 nhóm ~2.05 h
nhóm  29 | 1000 mẫu | lỗi đọc 0 | 267.2s (đọc 1s | ASR 264s) | còn 25 nhóm ~1.91 h
Zip 7: sẵn sàng (5,000 file giải nén, 22s) | ổ local trống 63.6 GB
nhóm  30 | 1000 mẫu | lỗi đọc 0 | 282.5s (đọc 1s | ASR 279s) | còn 24 nhóm ~1.85 h
nhóm  31 | 1000 mẫu | lỗi đọc 0 | 278.5s (đọc 1s | ASR 275s) | còn 23 nhóm ~1.78 h
nhóm  32 | 1000 mẫu | lỗi đọc 0 | 270.0s (đọc 1s | ASR 267s) | còn 22 nhóm ~1.69 h
nhóm  33 | 1000 mẫu | lỗi đọc 0 | 275.1s (đọc 1s | ASR 272s) | còn 21 nhóm ~1.61 h
nhóm  34 | 1000 mẫu | lỗi đọc 0 | 276.1s (đọc 1s | ASR 273s) | còn 20 nhóm ~1.53 h
Zip 8: sẵn sàng (5,000 file giải nén, 32s) | ổ local trống 61.8 GB
nhóm  35 | 1000 mẫu | lỗi đọc 0 | 276.2s (đọc 2s | ASR 272s) | còn 19 nhóm ~1.46 h
nhóm  36 | 1000 mẫu | lỗi đọc 0 | 266.8s (đọc 1s | ASR 264s) | còn 18 nhóm ~1.38 h
nhóm  37 | 1000 mẫu | lỗi đọc 0 | 279.7s (đọc 1s | ASR 276s) | còn 17 nhóm ~1.30 h
nhó

In [ ]:
# 4) Gộp các nhóm đã xong -> asr_hyp.csv và kiểm tra
parts = [pd.read_csv(chunk_path(k), keep_default_na=False) for k in range(N_CHUNKS) if chunk_ok(k)]
assert parts, 'Chưa có nhóm nào.'
allr = pd.concat(parts, ignore_index=True)
allr['asr_text'] = allr['asr_text'].astype(str)
pos = order.set_index('id')
allr['pos'] = pos.loc[allr.id, 'pos'].values; allr['split'] = pos.loc[allr.id, 'split'].values
assert allr.id.is_unique
print(f'Đã có {len(allr):,}/{TOTAL:,} mẫu | dev {int((allr.split=="dev").sum()):,} | train {int((allr.split=="train").sum()):,}')
print(f'Lỗi đọc audio: {int((allr.error != "").sum())} | ASR rỗng: {int((allr.asr_text.str.strip() == "").sum())} | '
      f'audio dài hơn {MAX_SEC}s (bị cắt): {int((allr.dur_s > MAX_SEC).sum())}')
tmp = os.path.join(OUT_DIR, 'asr_hyp.csv.tmp'); allr.to_csv(tmp, index=False); os.replace(tmp, os.path.join(OUT_DIR, 'asr_hyp.csv'))
print('Đã ghi asr_hyp.csv. Ví dụ (vi_text gốc | asr_text):')
for _, r in allr.head(5).iterrows(): print(' -', r.vi_text, '|', r.asr_text)

Đã có 55,000/55,000 mẫu | dev 5,500 | train 49,500
Lỗi đọc audio: 0 | ASR rỗng: 0 | audio dài hơn 30s (bị cắt): 0
Đã ghi asr_hyp.csv. Ví dụ (vi_text gốc | asr_text):
 - bà biết tôi ghét nhất là những chuyện gian dối mà | bà biết tôi ghét nhất là những chuyện gian dối mà.
 - ờ đừng có đánh nhiều quá ừ đúng rồi đó | ờ đừng có đánh nhiều quá mệt đúng cái đó.
 - và điện cẩm linh đang tổ chức | và điện cẩm linh đang tổ chức.
 - tụi em đoán được nhưng mà không dám nói từ từ từ từ nói sau cùng rồi bây giờ tôi mời hai cái người mà gọi là | tụi em đoán được nhưng mà không dám nói từ tới tới tới nói sau một cùng rồi bây giờ tôi mời hai cái người mà gọi là.
 - đã tránh hạn chế việc tiếp cận các hợp kim | đã tránh hạn chế việc tiếp cận các hợp kim.


In [ ]:
# 5) (Chỉ trên Colab) Đẩy toàn bộ phần ghi lên Drive trước khi kết thúc phiên
try:
    from google.colab import drive
    drive.flush_and_unmount(); print('Đã flush Drive.')
except Exception as e:
    print('Bỏ qua flush:', e)

Đã flush Drive.
